# Build processed TEC dataset

This notebook downloads daily IONEX/GIM files into temporary Colab storage, parses them, aggregates TEC values by predefined regions, and saves processed regional time series as CSV and Parquet.

Raw IONEX files are not stored in Git or Google Drive.

In [ ]:
from pathlib import Path
import sys
import os
import shutil
import gzip
import re
from datetime import date, datetime, timedelta
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError

import numpy as np
import pandas as pd

In [ ]:
IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # If the repo is cloned into /content/tec-agent-project
    PROJECT_ROOT = Path("/content/tec-agent-project")
    DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/tec_agent_project")

else:
    # Local VS Code mode
    PROJECT_ROOT = Path.cwd()
    DRIVE_PROJECT_DIR = PROJECT_ROOT

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

TMP_DIR = Path("/content/tec_tmp") if IN_COLAB else PROJECT_ROOT / "data" / "cache" / "tec_tmp"
TMP_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DIR = DRIVE_PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("TMP_DIR:", TMP_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

In [ ]:
from tec_agents.data.regions import REGIONS, list_region_ids

print("Regions:")
for region_id in list_region_ids():
    region = REGIONS[region_id]
    print(
        f"{region_id:24s} "
        f"lat=[{region.lat_min}, {region.lat_max}], "
        f"lon=[{region.lon_min}, {region.lon_max}]"
    )

In [ ]:
def daterange(start: date, end: date):
    """Yield dates in half-open interval [start, end)."""
    current = start
    while current < end:
        yield current
        current += timedelta(days=1)


def gps_week(d: date | datetime) -> int:
    """Return GPS week number for a date or datetime."""
    if isinstance(d, datetime):
        d = d.date()
    gps_epoch = date(1980, 1, 6)
    return (d - gps_epoch).days // 7


def doy(d: date) -> int:
    """Return day of year."""
    return d.timetuple().tm_yday

In [ ]:
import subprocess
import gzip
import shutil
from pathlib import Path
from datetime import datetime, timedelta, date


def build_codg_url(d: date | datetime) -> str:
    if isinstance(d, datetime):
        dt = d
    else:
        dt = datetime(d.year, d.month, d.day)

    year = dt.year
    day_of_year = dt.timetuple().tm_yday
    week = gps_week(dt.date())

    filename = f"COD0OPSFIN_{year}{day_of_year:03d}0000_01D_01H_GIM.INX.gz"
    return f"https://igs.bkg.bund.de/root_ftp/IGSac/products/{week}/{filename}"


def download_codg_day(
    d: date | datetime,
    out_dir: str | Path = TMP_DIR,
    overwrite: bool = False,
) -> Path:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    url = build_codg_url(d)
    gz_path = out_dir / url.split("/")[-1]
    inx_path = gz_path.with_suffix("")

    if inx_path.exists() and not overwrite:
        return inx_path

    if gz_path.exists() and overwrite:
        gz_path.unlink()

    if inx_path.exists() and overwrite:
        inx_path.unlink()

    print(f"Downloading {pd.Timestamp(d).date()} -> {gz_path.name}")

    subprocess.check_call(["wget", "-q", "-O", str(gz_path), url])

    if not gz_path.exists() or gz_path.stat().st_size == 0:
        raise RuntimeError(f"Downloaded empty file: {gz_path}")

    with gzip.open(gz_path, "rb") as f_in, open(inx_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

    if not inx_path.exists() or inx_path.stat().st_size == 0:
        raise RuntimeError(f"Failed to unpack file: {inx_path}")

    return inx_path

In [ ]:
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr

_NUM_RE = re.compile(r"[-+]?\d+(?:\.\d+)?(?:[DEde][-+]?\d+)?")


def _nums_from_line(line: str, n: int | None = None) -> list[float]:
    txt = line[:60].replace("D", "E").replace("d", "E")
    nums = [float(x) for x in _NUM_RE.findall(txt)]
    return nums if n is None else nums[:n]


def parse_ionex_to_xarray(path: str | Path, keep_raw_int: bool = False) -> xr.DataArray:
    path = Path(path)
    lines = path.read_text(errors="ignore").splitlines()

    lat1 = lat2 = dlat = None
    lon1 = lon2 = dlon = None
    exponent = 0
    i = 0

    while i < len(lines):
        line = lines[i]

        if "END OF HEADER" in line:
            i += 1
            break

        if "LAT1 / LAT2 / DLAT" in line:
            lat1, lat2, dlat = _nums_from_line(line, 3)

        if "LON1 / LON2 / DLON" in line:
            lon1, lon2, dlon = _nums_from_line(line, 3)

        if "EXPONENT" in line:
            exponent = int(_nums_from_line(line, 1)[0])

        i += 1

    if any(x is None for x in [lat1, lat2, dlat, lon1, lon2, dlon]):
        raise ValueError("Could not read lat/lon grid from IONEX header.")

    lat = np.arange(lat1, lat2 + (dlat / 2), dlat)
    lon = np.arange(lon1, lon2 + (dlon / 2), dlon)

    def parse_epoch(line: str) -> datetime:
        vals = _nums_from_line(line, 6)
        y, mo, d, hh, mm, ss = map(int, vals)
        return datetime(y, mo, d, hh, mm, ss)

    times = []
    maps = []

    while i < len(lines):
        line = lines[i]

        if "START OF TEC MAP" in line:
            cur_epoch = None
            grid = np.full((len(lat), len(lon)), np.nan, dtype=float)
            i += 1

            while i < len(lines):
                line = lines[i]

                if "EPOCH OF CURRENT MAP" in line:
                    cur_epoch = parse_epoch(line)
                    i += 1
                    continue

                if "LAT/LON1/LON2/DLON/H" in line:
                    lat_v, lon1_v, lon2_v, dlon_v, h_v = _nums_from_line(line, 5)
                    lat_idx = int(np.argmin(np.abs(lat - lat_v)))

                    vals = []
                    i += 1

                    while i < len(lines):
                        nxt = lines[i]

                        if ("LAT/LON1/LON2/DLON/H" in nxt) or ("END OF TEC MAP" in nxt):
                            break

                        vals.extend(_nums_from_line(nxt))
                        i += 1

                    arr = np.array(vals[: len(lon)], dtype=float)

                    if arr.size < len(lon):
                        arr = np.pad(
                            arr,
                            (0, len(lon) - arr.size),
                            constant_values=np.nan,
                        )

                    grid[lat_idx, :] = arr
                    continue

                if "END OF TEC MAP" in line:
                    if cur_epoch is None:
                        raise ValueError("TEC MAP without EPOCH OF CURRENT MAP.")
                    times.append(cur_epoch)
                    maps.append(grid)
                    break

                i += 1

        i += 1

    if len(maps) == 0:
        raise ValueError(f"No TEC MAP blocks found in {path}")

    data = np.stack(maps, axis=0)

    scale = 10.0 ** exponent
    if not keep_raw_int:
        data = data * scale

    da = xr.DataArray(
        data,
        coords={
            "time": pd.to_datetime(times),
            "lat": lat,
            "lon": lon,
        },
        dims=("time", "lat", "lon"),
        name="vtec",
        attrs={
            "source_file": str(path),
            "ionex_exponent": exponent,
            "units": "TECU" if not keep_raw_int else f"raw*10^{exponent}",
            "product": "CODE GIM COD0OPSFIN via IGS/BKG mirror",
        },
    )

    return da

In [ ]:
def normalize_xarray_lon(da: xr.DataArray) -> xr.DataArray:
    if float(da.lon.max()) > 180:
        new_lon = (((da.lon + 180) % 360) - 180)
        da = da.assign_coords(lon=new_lon).sortby("lon")
    return da


def aggregate_regions_from_xarray(da: xr.DataArray) -> pd.DataFrame:
    da = da.sortby("lat")
    da = normalize_xarray_lon(da)

    result = {}

    for region_id, region in REGIONS.items():
        sub = da.sel(
            lat=slice(region.lat_min, region.lat_max),
            lon=slice(region.lon_min, region.lon_max),
        )

        regional_mean = sub.mean(dim=("lat", "lon"), skipna=True)
        result[region_id] = pd.Series(
            regional_mean.values,
            index=pd.to_datetime(regional_mean.time.values),
            name=region_id,
        )

    out = pd.DataFrame(result)
    out.index.name = "time"
    out = out.sort_index()

    return out

In [ ]:
START_DATE = date(2024, 3, 1)
END_DATE = date(2024, 4, 1)

OUTPUT_BASENAME = "tec_regions_2024_03_hourly"

print("Date range:", START_DATE, "to", END_DATE)
print("Output basename:", OUTPUT_BASENAME)

In [ ]:
test_day = date(2024, 3, 1)

inx_path = download_codg_day(test_day, TMP_DIR, overwrite=False)
print("Downloaded and unpacked:", inx_path)
print("Size MB:", inx_path.stat().st_size / 1024 / 1024)

da_test = parse_ionex_to_xarray(inx_path)
print(da_test)

df_test = aggregate_regions_from_xarray(da_test)
display(df_test.head())
display(df_test.describe().T)

In [ ]:
daily_region_tables = []
daily_arrays = []

for d in daterange(START_DATE, END_DATE):
    try:
        inx_path = download_codg_day(d, TMP_DIR, overwrite=False)
        da_day = parse_ionex_to_xarray(inx_path)

        da_day = da_day.sortby("lat")
        df_regions_day = aggregate_regions_from_xarray(da_day)

        daily_arrays.append(da_day)
        daily_region_tables.append(df_regions_day)

        print(
            f"OK {d}: "
            f"maps={da_day.sizes.get('time')}, "
            f"lat={da_day.sizes.get('lat')}, "
            f"lon={da_day.sizes.get('lon')}, "
            f"region_points={len(df_regions_day)}"
        )

    except Exception as exc:
        print(f"FAIL {d}: {exc}")

if not daily_region_tables:
    raise RuntimeError("No daily tables were built.")

df_regions = pd.concat(daily_region_tables).sort_index()
df_regions = df_regions[~df_regions.index.duplicated(keep="first")]

print("Final shape:", df_regions.shape)
df_regions.head()

In [ ]:
print("Time range:", df_regions.index.min(), "->", df_regions.index.max())
print("Columns:", list(df_regions.columns))
print("Missing values by column:")
print(df_regions.isna().sum())

display(df_regions.describe().T)

In [ ]:
import matplotlib.pyplot as plt

ax = df_regions[["midlat_europe", "highlat_north", "equatorial_atlantic"]].plot(
    figsize=(12, 4),
    grid=True,
    title="Regional mean TEC, March 2024",
)
ax.set_ylabel("TEC, TECU")
plt.show()

In [ ]:
csv_path = PROCESSED_DIR / f"{OUTPUT_BASENAME}.csv"
parquet_path = PROCESSED_DIR / f"{OUTPUT_BASENAME}.parquet"

df_regions.to_csv(csv_path, index=True)
df_regions.to_parquet(parquet_path)

print("Saved CSV:", csv_path)
print("Saved Parquet:", parquet_path)

print("CSV size MB:", csv_path.stat().st_size / 1024 / 1024)
print("Parquet size MB:", parquet_path.stat().st_size / 1024 / 1024)

In [ ]:
CLEAN_TMP = True

if CLEAN_TMP:
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    print("Temporary raw files removed:", TMP_DIR)
else:
    print("Temporary raw files kept:", TMP_DIR)

In [ ]:
from tec_agents.data.datasets import register_dataset, get_dataset_summary, get_region_series

register_dataset(
    dataset_ref="default",
    path=parquet_path,
    file_format="parquet",
)

summary = get_dataset_summary("default")
summary

In [ ]:
from tec_agents.tools.executor import build_default_executor

executor = build_default_executor(run_id="real_dataset_smoke")

ts_result = executor.call(
    "tec_get_timeseries",
    {
        "dataset_ref": "default",
        "region_id": "midlat_europe",
        "start": "2024-03-01",
        "end": "2024-04-01",
    },
)

threshold_result = executor.call(
    "tec_compute_high_threshold",
    {
        "series_id": ts_result["series_id"],
        "method": "quantile",
        "q": 0.9,
    },
)

intervals_result = executor.call(
    "tec_detect_high_intervals",
    {
        "series_id": ts_result["series_id"],
        "threshold_id": threshold_result["threshold_id"],
        "min_duration_minutes": 0,
        "merge_gap_minutes": 60,
    },
)

print("Threshold:", threshold_result)
print("Intervals:", intervals_result["n_intervals"])
intervals_result["intervals"][:5]